# سحب عينة تدقيق قابلة لإعادة الإنتاج · فواتير الموردين 2026

ينفّذ خطة المعاينة في `sampling_plan.txt` كما كُتبت: مجاميع المجتمع، ثم البنود الرئيسة، ثم اختيار منتظم
ببداية عشوائية من بذرة مسجَّلة. من يعيد التشغيل بالبذرة نفسها على الملف نفسه يحصل على الفواتير نفسها.

In [ ]:
# الإعداد: يجد الملفات بجوار الدفتر أو في المجلد الأعلى. في خدمة دفاتر مستضافة: ارفع الملفات بجوار الدفتر.
import csv, datetime, hashlib
from pathlib import Path
import pandas as pd

def find(name):
    for p in (Path("."), Path(".."), Path("files")):
        if (p / name).exists():
            return p / name
    raise FileNotFoundError(name + " : ضع الملف بجوار الدفتر")

def log_run(notebook, inputs, totals):
    """يضيف سطرًا إلى سجل التشغيل: متى، وعلى أي ملفات (بصمتها)، وبأي مجاميع رقابية."""
    stamp = datetime.datetime.now().isoformat(timespec="seconds")
    marks = ";".join(p.name + ":" + hashlib.sha256(p.read_bytes()).hexdigest()[:12] for p in inputs)
    new = not Path("run_log.csv").exists()
    with open("run_log.csv", "a", encoding="utf-8-sig", newline="") as f:
        w = csv.writer(f)
        if new:
            w.writerow(["run_at", "notebook", "inputs_sha256", "control_totals"])
        w.writerow([stamp, notebook, marks, totals])
    print("سُجّل التشغيل في run_log.csv")

In [ ]:
import random
SEED, SIZE, KEY_ITEM, LIMIT = 20261031, 60, 100000, 50000
inv_path = find("invoices_fy2026.csv")
inv = pd.read_csv(inv_path, dtype=str, keep_default_na=False)
inv["h"] = (inv["amount"].astype(float) * 100).round().astype("int64")
population = {"count": len(inv), "sum": inv["h"].sum() / 100}
key_items = inv[inv["h"] >= KEY_ITEM * 100]
rest = inv[inv["h"] < KEY_ITEM * 100].sort_values("invoice_id").reset_index(drop=True)
assert len(key_items) + len(rest) == len(inv) and key_items["h"].sum() + rest["h"].sum() == inv["h"].sum()
print("المجتمع:", population)
print("البنود الرئيسة:", len(key_items), key_items["h"].sum() / 100, "· الباقي:", len(rest), rest["h"].sum() / 100)

In [ ]:
interval = len(rest) // SIZE
start = int(random.Random(SEED).random() * interval) + 1          # أول عدد عشوائي من البذرة يحدد البداية
positions = [start + interval * k for k in range(SIZE)]           # مواضع تبدأ من 1
sample = rest.iloc[[p - 1 for p in positions]]
print("البذرة:", SEED, "· الفترة:", interval, "· البداية:", start, "· حجم العينة:", len(sample))
print("مجموع مبالغ العينة:", sample["h"].sum() / 100)
print(sample["invoice_id"].head(5).tolist(), "...", sample["invoice_id"].tail(2).tolist())

## الاختبارات الأربعة على الفواتير المفحوصة

ما يظهر هنا «بند للنظر». هل هو استثناء؟ هذا حكم المدقق بعد قراءة المستند.

In [ ]:
tested = pd.concat([key_items.assign(source="بند رئيس"), sample.assign(source="العينة")])
twin = inv.duplicated(["supplier_id", "supplier_invoice_no", "h"], keep=False)
period_end = pd.to_datetime(inv["posting_period"] + "-01") + pd.offsets.MonthEnd(0)
late = pd.to_datetime(inv["invoice_date"]) > period_end
under = inv[inv["h"] < LIMIT * 100]
g = under.groupby(["supplier_id", "invoice_date"])["h"].agg(["count", "sum"])
split_keys = set(g[(g["count"] > 1) & (g["sum"] >= LIMIT * 100)].index)
split = inv.apply(lambda r: r["h"] < LIMIT * 100 and (r["supplier_id"], r["invoice_date"]) in split_keys, axis=1)
flags = pd.DataFrame({"invoice_id": inv["invoice_id"], "بلا أمر شراء": inv["po_number"] == "", "لها فاتورة مطابقة": twin,
                      "تاريخها بعد فترة الترحيل": late, "تجزئة دون حد الاعتماد": split})
look = tested[["invoice_id", "source", "supplier_id", "amount"]].merge(flags, on="invoice_id")
look = look[look.iloc[:, 4:].any(axis=1)]
print(look.to_string(index=False))

In [ ]:
out = tested[["invoice_id", "source", "supplier_id", "supplier_invoice_no", "invoice_date", "amount", "po_number"]].merge(flags, on="invoice_id")
out["حكم المدقق"] = ""
out.to_csv("sampling_file.csv", index=False, encoding="utf-8-sig")
with open("sampling_file.csv", "a", encoding="utf-8-sig", newline="") as f:
    f.write(f"# method=systematic with random start; seed={SEED}; interval={interval}; start={start}; size={SIZE}; "
            f"population={population['count']}/{population['sum']:.2f}; key_items={len(key_items)}/{key_items['h'].sum() / 100:.2f}\n")
log_run("audit_sample", [inv_path],
        f"population {population['count']}/{population['sum']:.2f}; key {len(key_items)}; seed {SEED}; start {start}; interval {interval}; "
        f"sample {len(sample)}/{sample['h'].sum() / 100:.2f}; to_look_at {len(look)}")